In [ ]:
# gtcc +svm +spread_spectrum
import numpy as np
import os
import pickle
import hashlib
import librosa
import soundfile as sf
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives.serialization import (
    Encoding, PrivateFormat, PublicFormat, NoEncryption
)
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import joblib
        
class SpreadSpectrumWatermarker:
    """Audio watermarking class implementing Spread Spectrum technique"""
    
    def __init__(self, watermark_bits=32, alpha=0.1, frame_size=1024):
        """
        Initialize the Spread Spectrum watermarker
        
        Args:
            watermark_bits: Number of bits in the watermark
            alpha: Watermark strength factor (controls imperceptibility vs. robustness)
            frame_size: Size of audio windows for embedding each bit
        """
        self.watermark_bits = watermark_bits
        self.alpha = alpha
        self.frame_size = frame_size
        
        # Generate standard watermark sequence - the same for all files
        self.standard_watermark = np.random.randint(0, 2, size=watermark_bits).astype(np.int8)
        
        # Generate pseudo-random spreading sequences for each bit
        self.spreading_sequences = []
        for i in range(watermark_bits):
            # Create a pseudorandom sequence for each bit
            np.random.seed(i)  # Different seed for each bit for orthogonality
            spreading_seq = np.random.randn(frame_size)
            # Normalize the spreading sequence
            spreading_seq = spreading_seq / np.sqrt(np.sum(spreading_seq**2))
            self.spreading_sequences.append(spreading_seq)
    
    def embed(self, audio, sr=16000):
        """Embed standard watermark into audio using Spread Spectrum technique"""
        
        watermarked_audio = audio.copy()
        
        # Make sure audio is long enough
        required_length = self.watermark_bits * self.frame_size
        if len(audio) < required_length:
            padding = np.zeros(required_length - len(audio))
            watermarked_audio = np.concatenate([watermarked_audio, padding])
        
        # Embed each bit of the watermark
        for i in range(self.watermark_bits):
            # Get the bit and its spreading sequence
            bit = self.standard_watermark[i]
            spreading_seq = self.spreading_sequences[i]
            
            # Modify bit value from {0,1} to {-1,1}
            bit_value = 2 * bit - 1  # 0 -> -1, 1 -> 1
            
            # Get the frame from audio
            start_idx = i * self.frame_size
            end_idx = start_idx + self.frame_size
            
            if end_idx > len(watermarked_audio):
                break  # Avoid out of bounds
            
            # Embed the watermark bit using spread spectrum
            watermarked_frame = watermarked_audio[start_idx:end_idx]
            watermarked_frame += self.alpha * bit_value * spreading_seq
            
            # Update the audio
            watermarked_audio[start_idx:end_idx] = watermarked_frame
        
        # Normalize to prevent clipping
        max_val = np.max(np.abs(watermarked_audio))
        if max_val > 1.0:
            watermarked_audio = watermarked_audio / max_val
        
        return watermarked_audio
    
    def detect(self, audio, sr=16000):
        """Detect standard watermark in audio using Spread Spectrum"""
        # Initialize extracted bits
        extracted_bits = np.zeros(self.watermark_bits, dtype=np.int8)
        correlation_values = np.zeros(self.watermark_bits)
        
        # For each bit
        for i in range(self.watermark_bits):
            # Get the spreading sequence for this bit
            spreading_seq = self.spreading_sequences[i]
            
            # Get the corresponding frame from audio
            start_idx = i * self.frame_size
            end_idx = start_idx + self.frame_size
            
            if end_idx > len(audio):
                break  # Avoid out of bounds
            
            frame = audio[start_idx:end_idx]
            
            # Calculate correlation between frame and spreading sequence
            correlation = np.sum(frame * spreading_seq)
            correlation_values[i] = correlation
            
            # Extract bit based on correlation sign
            extracted_bits[i] = 1 if correlation > 0 else 0
        
        # Calculate similarity with standard watermark
        similarity = np.sum(extracted_bits == self.standard_watermark) / self.watermark_bits
        
        # Return True if similarity exceeds threshold (70%)
        return similarity > 0.7, similarity    


class DigitalSignature:
    """Class for digital signature operations"""
    
    def __init__(self):
        # Generate RSA key pair
        self.private_key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=2048
        )
        self.public_key = self.private_key.public_key()
    
    def generate_content_hash(self, audio):
        """Generate a hash of audio content to serve as a unique identifier"""
        # Normalize to avoid minor variations
        audio_norm = audio / (np.max(np.abs(audio)) + 1e-8)
        # Use a coarse representation to be more stable
        audio_int = np.int16(audio_norm * 32767)
        # Use subsampling for more stability against minor modifications
        subsampled = audio_int[::10]  # Take every 10th sample
        audio_bytes = subsampled.tobytes()
        return hashlib.sha256(audio_bytes).hexdigest()
    
    def generate_signature_hash(self, audio):
        """Generate SHA-256 hash of audio data for signature verification"""
        # This should be more precise than content hash
        audio_norm = audio / (np.max(np.abs(audio)) + 1e-8)
        audio_int = np.int16(audio_norm * 32767)
        audio_bytes = audio_int.tobytes()
        return hashlib.sha256(audio_bytes).digest()
    
    def sign(self, audio):
        """Sign audio with private key"""
        audio_hash = self.generate_signature_hash(audio)
        signature = self.private_key.sign(
            audio_hash,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        return signature
    
    def verify(self, audio, signature):
        """Verify signature with public key"""
        audio_hash = self.generate_signature_hash(audio)
        try:
            self.public_key.verify(
                signature,
                audio_hash,
                padding.PSS(
                    mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.MAX_LENGTH
                ),
                hashes.SHA256()
            )
            return True
        except Exception as e:
            print(f"Signature verification failed: {e}")
            return False
    
    def export_keys(self):
        """Export keys for later use"""
        private_key_bytes = self.private_key.private_bytes(
            encoding=Encoding.PEM,
            format=PrivateFormat.PKCS8,
            encryption_algorithm=NoEncryption()
        )
        public_key_bytes = self.public_key.public_bytes(
            encoding=Encoding.PEM,
            format=PublicFormat.SubjectPublicKeyInfo
        )
        return private_key_bytes, public_key_bytes


class FeatureExtractor:
    """Class for extracting features from audio for spoofing detection"""
    
    def __init__(self, n_mfcc=20, n_gtcc=20, n_mels=128, hop_length=512, 
                 use_mfcc=True, use_gtcc=False, use_spectral=False):
        self.n_mfcc = n_mfcc
        self.n_gtcc = n_gtcc
        self.n_mels = n_mels
        self.hop_length = hop_length
        
        # Feature selection flags
        self.use_mfcc = use_mfcc
        self.use_gtcc = use_gtcc
        self.use_spectral = use_spectral
    
    def extract_mfcc(self, audio, sr=16000):
        """Extract Mel Frequency Cepstral Coefficients"""
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=self.n_mfcc, hop_length=self.hop_length)
        # Compute statistics over time
        mfcc_mean = np.mean(mfccs, axis=1)
        mfcc_std = np.std(mfccs, axis=1)
        mfcc_delta = librosa.feature.delta(mfccs)
        mfcc_delta_mean = np.mean(mfcc_delta, axis=1)
        mfcc_delta_std = np.std(mfcc_delta, axis=1)
        
        return np.concatenate([mfcc_mean, mfcc_std, mfcc_delta_mean, mfcc_delta_std])
    
    def extract_gtcc(self, audio, sr=16000):
        """Extract Gammatone Cepstral Coefficients"""
        # Using librosa's mel spectrogram as an approximation for GTCC
        # A proper GTCC implementation would use a gammatone filterbank
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=self.n_gtcc, hop_length=self.hop_length)
        log_mel = librosa.power_to_db(mel_spec)
        gtccs = librosa.feature.mfcc(S=log_mel, n_mfcc=self.n_gtcc)
        
        # Compute statistics
        gtcc_mean = np.mean(gtccs, axis=1)
        gtcc_std = np.std(gtccs, axis=1)
        gtcc_delta = librosa.feature.delta(gtccs)
        gtcc_delta_mean = np.mean(gtcc_delta, axis=1)
        gtcc_delta_std = np.std(gtcc_delta, axis=1)
        
        return np.concatenate([gtcc_mean, gtcc_std, gtcc_delta_mean, gtcc_delta_std])
    
    def extract_spectrogram_features(self, audio, sr=16000):
        """Extract features from spectrogram"""
        # Compute spectrogram
        spec = np.abs(librosa.stft(audio, hop_length=self.hop_length))
        
        # Extract spectral statistics
        spec_centroid = librosa.feature.spectral_centroid(S=spec)[0]
        spec_bandwidth = librosa.feature.spectral_bandwidth(S=spec)[0]
        spec_rolloff = librosa.feature.spectral_rolloff(S=spec)[0]
        spec_contrast = librosa.feature.spectral_contrast(S=spec)[0]
        
        # Handle potential empty arrays
        mean_contrast = np.mean(spec_contrast) if spec_contrast.size > 0 else 0
        std_contrast = np.std(spec_contrast) if spec_contrast.size > 0 else 0
        
        # Compute statistics
        features = np.concatenate([
            [np.mean(spec_centroid), np.std(spec_centroid)],
            [np.mean(spec_bandwidth), np.std(spec_bandwidth)],
            [np.mean(spec_rolloff), np.std(spec_rolloff)],
            [mean_contrast, std_contrast]
        ])
        
        return features
    
    def extract_all_features(self, audio, sr=16000):
        """Extract selected features from audio based on configuration"""
        try:
            # Make sure audio isn't too short
            if len(audio) < sr:  # Less than 1 second
                padding = np.zeros(sr - len(audio))
                audio = np.concatenate([audio, padding])
            
            # Extract features based on configuration
            features_list = []
            
            if self.use_mfcc:
                mfcc_features = self.extract_mfcc(audio, sr)
                features_list.append(mfcc_features)
            
            if self.use_gtcc:
                gtcc_features = self.extract_gtcc(audio, sr)
                features_list.append(gtcc_features)
            
            if self.use_spectral:
                spec_features = self.extract_spectrogram_features(audio, sr)
                features_list.append(spec_features)
            
            # Combine all selected features
            if features_list:
                all_features = np.concatenate(features_list)
                return all_features
            else:
                # If no features are selected, use MFCC as fallback
                print("Warning: No features selected. Using MFCC as fallback.")
                return self.extract_mfcc(audio, sr)
                
        except Exception as e:
            print(f"Error in feature extraction: {e}")
            # Return a feature vector of zeros with the expected length
            expected_length = 0
            if self.use_mfcc:
                expected_length += self.n_mfcc * 4
            if self.use_gtcc:
                expected_length += self.n_gtcc * 4
            if self.use_spectral:
                expected_length += 8  # Approximate length of spectral features
            
            if expected_length == 0:
                expected_length = self.n_mfcc * 4  # Fallback
            
            return np.zeros(expected_length)


class SpoofingDetectionModel:
    """Machine learning model for spoofing detection"""
    
    def __init__(self, model_type='svm'):
        self.model_type = model_type
        if model_type == 'rf':
            self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        elif model_type == 'svm':
            self.model = SVC(probability=True, random_state=42)
        elif model_type == 'xgb':
            from xgboost import XGBClassifier
            self.model = XGBClassifier(use_label_encoder=False, random_state=42)
        else:
            raise ValueError(f"Unsupported model type: {model_type}")
        
        self.scaler = StandardScaler()
    
    def train(self, X, y):
        """Train the model on extracted features"""
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Split data into training and validation sets
        X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
        
        # Train model
        self.model.fit(X_train, y_train)
        
        # Evaluate model
        val_acc = self.model.score(X_val, y_val)
        y_pred = self.model.predict(X_val)
        report = classification_report(y_val, y_pred, output_dict=True)
        
        print(f"Validation accuracy: {val_acc:.4f}")
        print(f"Real audio F1-score: {report['1']['f1-score']:.4f}")
        print(f"Spoofed audio F1-score: {report['0']['f1-score']:.4f}")
        
        return val_acc
    
    def predict(self, X):
        """Predict if audio is real or spoofed"""
        # Scale features
        X_scaled = self.scaler.transform(X.reshape(1, -1))
        
        # Predict
        prediction = self.model.predict(X_scaled)
        probability = self.model.predict_proba(X_scaled)[0]
        
        return bool(prediction[0]), probability
    
    def save(self, path):
        """Save model to file"""
        joblib.dump((self.model, self.scaler), path)
        print(f"Model saved to {path}")
    
    def load(self, path):
        """Load model from file"""
        self.model, self.scaler = joblib.load(path)
        print(f"Model loaded from {path}")


class AudioPreprocessor:
    """Class for preprocessing audio with spread spectrum watermarking and signature database"""
    
    def __init__(self, watermark_bits=64, alpha=0.1):
        self.watermarker = SpreadSpectrumWatermarker(
            watermark_bits=watermark_bits,
            alpha=alpha,
        )
        self.digital_signature = DigitalSignature()
        
        # Database to store signatures indexed by content hash
        self.signature_db = {}
    
    def process_directory(self, input_dir, output_dir, sample_rate=16000):
        """Process all audio files in a directory"""
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Process each audio file
        processed_count = 0
        for filename in os.listdir(input_dir):
            if filename.endswith(('.wav', '.mp3', '.flac')):
                try:
                    # Load audio file
                    filepath = os.path.join(input_dir, filename)
                    audio, sr = librosa.load(filepath, sr=sample_rate, mono=True)
                    
                    # Process audio
                    processed_audio = self.process_audio(audio)
                    
                    output_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}_processed.wav")
                    sf.write(output_path, processed_audio, sample_rate)
                    
                    processed_count += 1
                    print(f"Processed {filename} -> {output_path}")
                except Exception as e:
                    print(f"Error processing {filename}: {e}")
        
        # Save signature database
        self.save_signature_db(os.path.join(output_dir, "signature_db.pkl"))
        
        print(f"Processed {processed_count} files.")
        return processed_count
    
    def process_audio(self, audio):
        """Process a single audio file with spread spectrum watermarking and digital signature"""
        # 1. Embed spread spectrum watermark
        watermarked_audio = self.watermarker.embed(audio)
        
        # 2. Generate content hash for identification
        content_hash = self.digital_signature.generate_content_hash(watermarked_audio)
        
        # 3. Generate digital signature
        signature = self.digital_signature.sign(watermarked_audio)
        
        # 4. Store signature in database
        self.signature_db[content_hash] = signature
        
        return watermarked_audio
    
    def verify_audio(self, audio):
        """Verify audio using watermark detection and signature lookup"""
        # 1. Check if standard watermark is present
        watermark_present, similarity = self.watermarker.detect(audio)
        
        if not watermark_present:
            print("Watermark not detected - Audio classified as SPOOFED")
            return False, 0.0, "No watermark"
        
        # 2. Generate content hash
        content_hash = self.digital_signature.generate_content_hash(audio)
        
        # 3. Look up signature in database
        if content_hash in self.signature_db:
            # 4. Verify signature
            signature_valid = self.digital_signature.verify(
                audio, self.signature_db[content_hash]
            )
            
            if signature_valid:
                print(f"Audio verified successfully (watermark similarity: {similarity:.2f})")
                return True, similarity, "Verified"
            else:
                print(f"Digital signature verification failed (watermark similarity: {similarity:.2f})")
                return False, similarity, "Invalid signature"
        else:
            print(f"Content hash not found in signature database (watermark similarity: {similarity:.2f})")
            return False, similarity, "Unknown content"
    
    def save_signature_db(self, filename):
        """Save the signature database"""
        with open(filename, 'wb') as f:
            pickle.dump({
                'signature_db': self.signature_db,
                'watermark_bits': self.watermarker.watermark_bits,
                'alpha': self.watermarker.alpha,
                'standard_watermark': self.watermarker.standard_watermark,
                'spreading_sequences': self.watermarker.spreading_sequences,
                'keys': self.digital_signature.export_keys()
            }, f)
        print(f"Signature database saved to {filename}")
    
    def load_signature_db(self, filename):
        """Load the signature database"""
        with open(filename, 'rb') as f:
            data = pickle.load(f)
            self.signature_db = data['signature_db']
            self.watermarker.watermark_bits = data['watermark_bits']
            self.watermarker.alpha = data['alpha']
            self.watermarker.standard_watermark = data['standard_watermark']
            self.watermarker.spreading_sequences = data['spreading_sequences']
            
        print(f"Signature database loaded from {filename}")


class AudioSpoofingDetectionSystem:
    """Improved audio spoofing detection system that combines spread spectrum watermarking, 
    signature database, and ML-based detection"""
    
    def __init__(self, watermark_bits=64, alpha=0.1, model_type='svm',
                use_mfcc=True, use_gtcc=False, use_spectral=False):
        # Initialize components
        self.preprocessor = AudioPreprocessor(
            watermark_bits=watermark_bits, 
            alpha=alpha
        )
        self.feature_extractor = FeatureExtractor(
            use_mfcc=use_mfcc,
            use_gtcc=use_gtcc,
            use_spectral=use_spectral
        )
        self.ml_model = SpoofingDetectionModel(model_type=model_type)
        
        # Status flags
        self.is_trained = False
        
        # Feature configuration for reporting
        self.feature_config = {
            'mfcc': use_mfcc,
            'gtcc': use_gtcc,
            'spectral': use_spectral
        }
    
    def train(self, bonafide_dir, spoofed_dir, sample_rate=16000):
        """Train the complete system"""
        print("=== Starting Training Phase ===")
        print("1. Processing bonafide audio files (watermarking + signatures)...")
        
        # Print feature configuration
        print(f"Feature configuration: MFCC: {self.feature_config['mfcc']}, " 
              f"GTCC: {self.feature_config['gtcc']}, " 
              f"Spectral: {self.feature_config['spectral']}")
        
        # Create a directory for processed bonafide audio
        processed_dir = os.path.join(os.path.dirname(bonafide_dir), "processed_real_ss")
        self.preprocessor.process_directory(bonafide_dir, processed_dir, sample_rate)
        
        print("\n2. Extracting features for ML model...")
        
        # Prepare feature data and labels
        features = []
        labels = []
        
        # Process bonafide audio (labeled as 1)
        print("  Extracting features from bonafide audio...")
        for filename in os.listdir(processed_dir):
            if filename.endswith('.wav'):
                try:
                    filepath = os.path.join(processed_dir, filename)
                    audio, sr = librosa.load(filepath, sr=sample_rate, mono=True)
                    
                    # Extract features
                    audio_features = self.feature_extractor.extract_all_features(audio, sr)
                    features.append(audio_features)
                    labels.append(1)  # 1 = real
                except Exception as e:
                    print(f"Error extracting features from {filename}: {e}")
        
        # Process spoofed audio (labeled as 0)
        print("  Extracting features from spoofed audio...")
        for filename in os.listdir(spoofed_dir):
            if filename.endswith(('.wav', '.mp3', '.flac')):
                try:
                    filepath = os.path.join(spoofed_dir, filename)
                    audio, sr = librosa.load(filepath, sr=sample_rate, mono=True)
                    
                    # Extract features
                    audio_features = self.feature_extractor.extract_all_features(audio, sr)
                    features.append(audio_features)
                    labels.append(0)  # 0 = spoofed
                except Exception as e:
                    print(f"Error extracting features from {filename}: {e}")
        
        # Convert to numpy arrays
        X = np.array(features)
        y = np.array(labels)
        
        # Train ML model
        print("\n3. Training ML model...")
        validation_accuracy = self.ml_model.train(X, y)
        
        # Save model
        models_dir = os.path.join(os.path.dirname(bonafide_dir), "models")
        os.makedirs(models_dir, exist_ok=True)
        self.ml_model.save(os.path.join(models_dir, "spoofing_detection_model.pkl"))
        
        self.is_trained = True
        print(f"\n=== Training Complete (Validation Accuracy: {validation_accuracy:.4f}) ===")
        
        return validation_accuracy
    
    def detect_spoofing(self, audio_path, sample_rate=16000):
        """Detect if an audio file is spoofed using the improved pipeline"""
        if not self.is_trained and not os.path.exists("models/spoofing_detection_model.pkl"):
            print("Error: System not trained. Please train the system first.")
            return False, 0.0
        
        print("=== Starting Detection ===")
        
        # Load audio
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
        
        # Step 1: Verify security features (watermark and signature)
        print("1. Verifying security features...")
        security_passed, similarity, reason = self.preprocessor.verify_audio(audio)
        
        if security_passed:
            print(f"Security verification passed - Audio classified as REAL ({reason})")
            return True, 1.0  # High confidence of being real
        
        # Step 2: If security verification failed, run ML detection as backup
        print("2. Security verification failed - Running ML detection...")
        
        # Extract features
        print("3. Extracting audio features...")
        features = self.feature_extractor.extract_all_features(audio, sr)
        
        # Run ML classification
        print("4. Running ML classification...")
        if not self.is_trained:
            self.ml_model.load("models/spoofing_detection_model.pkl")
            self.is_trained = True
            
        is_real, probabilities = self.ml_model.predict(features)
        
        # Final decision based on ML model (security already failed)
        spoofed_probability = probabilities[0] if not is_real else 1.0 - probabilities[1]
        confidence = (1.0 - spoofed_probability) if is_real else spoofed_probability
        
        result = "REAL" if is_real else "SPOOFED"
        print(f"=== Detection Result: {result} (Confidence: {confidence:.4f}, Security: {reason}) ===")
        
        return is_real, confidence
    
    def load_system(self, signature_db_path, model_path):
        """Load a previously trained system"""
        # Load signature database
        self.preprocessor.load_signature_db(signature_db_path)
        
        # Load ML model
        self.ml_model.load(model_path)
        self.is_trained = True
        
        print("System loaded successfully")
        
    def save_system(self, output_dir):
        """Save the complete system"""
        os.makedirs(output_dir, exist_ok=True)
        
        # Save signature database
        signature_db_path = os.path.join(output_dir, "signature_db.pkl")
        self.preprocessor.save_signature_db(signature_db_path)
        
        # Save ML model
        model_path = os.path.join(output_dir, "spoofing_detection_model.pkl")
        self.ml_model.save(model_path)
        
        # Save feature configuration
        feature_config_path = os.path.join(output_dir, "feature_config.pkl")
        with open(feature_config_path, 'wb') as f:
            pickle.dump(self.feature_config, f)
        
        print(f"System saved to {output_dir}")


# Example usage
if __name__ == "__main__":
    # Feature configuration
    
    print("Using only MFCC features for training and testing")
    
    USE_MFCC = True       
    USE_GTCC = False
    USE_SPECTRAL = False
    
    # Initialize the system with feature selection
    system = AudioSpoofingDetectionSystem(
        watermark_bits=64,
        alpha=0.1,  # Spread spectrum strength factor
        model_type='svm',
        use_mfcc=USE_MFCC,
        use_gtcc=USE_GTCC,
        use_spectral=USE_SPECTRAL
    )
    
    # Define real and fake subdirectories
    train_real_dir = "./archive/train/real/"
    train_fake_dir = "./archive/train/fake/"
    
    # Training phase
    system.train(
        bonafide_dir=train_real_dir,
        spoofed_dir=train_fake_dir,
        sample_rate=16000
    )
    
    # Save the trained system
    system.save_system("trained_system")
    
    # Process evaluation real files with watermarks
    print("\n=== Processing Evaluation Real Files with Spread Spectrum Watermarks ===")
    
    eval_real_processed_dir = "./archive/eval/processed_real_ss"
    os.makedirs(eval_real_processed_dir, exist_ok=True)
    system.preprocessor.process_directory(
        input_dir="./archive/eval/real",
        output_dir=eval_real_processed_dir,
        sample_rate=16000
    )
    
    # Testing phase - process all files in test directory
    test_results = []
    
    # Test real samples (using processed files)
    print("\n=== Testing Real Audio Samples ===")
    test_real_dir = eval_real_processed_dir  # Use the processed directory
    for filename in os.listdir(test_real_dir):
        if filename.endswith(('.wav', '.mp3', '.flac')):
            audio_path = os.path.join(test_real_dir, filename)
            is_real, confidence = system.detect_spoofing(
                audio_path=audio_path,
                sample_rate=16000
            )
            test_results.append({
                'file': filename,
                'actual': 'REAL',
                'predicted': 'REAL' if is_real else 'SPOOFED',
                'confidence': confidence
            })
            print(f"{filename} is {'REAL' if is_real else 'SPOOFED'} with {confidence*100:.2f}% confidence")
    
    # Test fake samples
    print("\n=== Testing Fake Audio Samples ===")
    test_fake_dir = "./archive/eval/fake/"
    for filename in os.listdir(test_fake_dir):
        if filename.endswith(('.wav', '.mp3', '.flac')):
            audio_path = os.path.join(test_fake_dir, filename)
            is_real, confidence = system.detect_spoofing(
                audio_path=audio_path,
                sample_rate=16000
            )
            test_results.append({
                'file': filename,
                'actual': 'FAKE',
                'predicted': 'REAL' if is_real else 'SPOOFED',
                'confidence': confidence
            })
            print(f"{filename} is {'Real' if is_real else 'SPOOFED'} with {confidence*100:.2f}% confidence")
    # Calculate overall accuracy
    correct = sum(1 for result in test_results if 
                  (result['actual'] == 'REAL' and result['predicted'] == 'REAL') or
                  (result['actual'] == 'FAKE' and result['predicted'] == 'SPOOFED'))
    accuracy = correct / len(test_results) if test_results else 0
    print(f"\nOverall Testing Accuracy: {accuracy*100:.2f}%")
    # Save results to CSV
    import csv
    with open('new_test_gtcc_svm_spread.csv', 'w', newline='') as csvfile:
        fieldnames = ['file', 'actual', 'predicted', 'confidence']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for result in test_results:
            writer.writerow(result)
            
